# 22 — The Jury: how much do 20 equally good detectors disagree?

**Goal.** Twenty detectors (seeds 42–61) were trained on the same essays with the same recipe. This notebook measures how often they disagree, on whom, and how far that should change what we say on the web page.
The official verdict stays the pre-declared three-seed ensemble; the jury shows how far to trust it.

**Rules fixed before computing** (`docs/jury_phase2_declaration.md`, commit `8b0b7ec`): the statistics S0–S10 below, their result files, and the rule that a juror whose fp16 and fp32 zones differ on more than 10 essays is served from fp32.
Held-out prompts and generators are **descriptive**: nothing is chosen from them. Nothing here changes the final system or its thresholds.

**Vocabulary.**
- A *juror's logit* is its raw score for an essay. Its *p* is the share of its own 2,291 validation humans that score at least as high (plus one), so p is small for AI-looking essays.
- A juror's *zone*: **flagged** if p ≤ 0.00235, **above usual** if p ≤ 0.01032, otherwise **not flagged**. These taus came from single-seed folds in 18b, so they are meant for single models.
- *Group A* = seeds 42–44 (also the official ensemble); *group B* = seeds 45–61 (17 jurors that are not).

In [1]:
import sys, json, warnings
sys.path.append('..')
import numpy as np, pandas as pd
from scipy.stats import rankdata
from src.data import load_splits
from src.conformal import tail_p
from src.ensemble import PEnsemble
from src.evaluate import wilson_interval
warnings.filterwarnings('ignore')
R = '../results/'
TAU_FLAG, TAU_ABOVE = 0.00235, 0.01032
SEEDS = list(range(42, 62)); COLS = [f's{s}' for s in SEEDS]
GROUPS = {'all20': list(range(20)), 'A_42-44': [0, 1, 2], 'B_45-61': list(range(3, 20))}
data = load_splits()
meta = data[['split', 'label', 'prompt_name', 'dup_cluster']]
vt = pd.read_csv(R + '22_juror_scores.csv').set_index('row_id')[COLS]
ho = pd.read_csv(R + '22_juror_scores_heldout.csv').set_index('row_id')[COLS]
L = pd.concat([vt, ho])                                   # logits of the fp16 serving copies, one column per juror
M = meta.loc[L.index]
assert len(L) == 7002 + 7135 + 2721 and L.notna().all().all()
print(L.shape, M.split.value_counts().to_dict())
summary = []          # long format rows -> results/22_jury_summary.csv
def add(stat, pop, group, metric, value, lo=np.nan, hi=np.nan, n=np.nan):
    summary.append(dict(stat=stat, population=pop, group=group, metric=metric, value=value, lo=lo, hi=hi, n=n))
def rate(k, n):
    lo, hi = wilson_interval(int(k), int(n)); return k / n, lo, hi

(16858, 20) {'heldout_prompt': 7135, 'test': 3501, 'val': 3501, 'heldout_generator': 2721}


## 1. Each juror's zones, from its own validation humans

Step: for every juror, take the logits of the 2,291 validation humans and turn any essay's logit into its tail probability `p` (`tail_p`), then into a zone.
The flag line is therefore *per juror*: it sits where at most 4 of that juror's validation humans score as high. We check the cut-offs against the Phase 1 table.

In [2]:
val_h = (M.split == 'val') & (M.label == 0)
V = {c: L.loc[val_h[val_h].index, c].to_numpy() for c in COLS}
assert all(len(v) == 2291 for v in V.values())
P = np.column_stack([tail_p(L[c].to_numpy(), V[c]) for c in COLS])      # (essays, 20) tail probabilities
Z = np.where(P <= TAU_FLAG, 2, np.where(P <= TAU_ABOVE, 1, 0))          # 2 flagged, 1 above usual, 0 not flagged
FL = (Z == 2)
jt = pd.read_csv(R + '22_jurors.csv').set_index('seed')
# exact flag line: the (floor(tau*(1+n)))-th highest validation-human logit; flagged = strictly above it (the Phase 1 table rounds it to 3 decimals)
flag_line = np.array([np.sort(V[c])[::-1][int(np.floor(TAU_FLAG * (1 + len(V[c])))) - 1] for c in COLS])
assert np.allclose(flag_line, [jt.loc[s, 'flag_logit_cutoff_tau1pct'] for s in SEEDS], atol=6e-4)
assert ((L.to_numpy() > flag_line) == FL).all()          # "logit above the line" is the same as "p <= tau" 
print('flag lines match Phase 1 table; share of val humans flagged by each juror (should be <= 0.17%):')
print((FL[(val_h).to_numpy()].mean(0) * 100).round(3))

flag lines match Phase 1 table; share of val humans flagged by each juror (should be <= 0.17%):
[0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175
 0.175 0.175 0.175 0.175 0.175 0.175 0.175 0.175]


## 2. S0 — does serving in fp16 change anyone's zone? (rule fixed in advance)

The fp32 path is the training path (fp32 weights, fp16 autocast). For each juror we calibrate it separately on its own validation humans and compare the zone of every one of the 7,002 validation and test essays. Rule: **more than 10 differing essays → that juror is served from fp32.**
Secondary check: fp32 scores judged with the *fp16* cut-offs, to see how much of a difference is calibration and how much the scores.

In [3]:
F = pd.read_csv(R + '22_juror_scores_fp32path.csv').set_index('row_id')[COLS]
assert (F.index == vt.index).all()
mv = (M.loc[F.index].split == 'val') & (M.loc[F.index].label == 0)
rows = []
for j, c in enumerate(COLS):
    pf = tail_p(F[c].to_numpy(), F.loc[mv[mv].index, c].to_numpy())
    z32 = np.where(pf <= TAU_FLAG, 2, np.where(pf <= TAU_ABOVE, 1, 0))
    z16 = Z[:len(F), j]
    p_cross = tail_p(F[c].to_numpy(), V[c])                               # fp32 scores, fp16 calibration
    zc = np.where(p_cross <= TAU_FLAG, 2, np.where(p_cross <= TAU_ABOVE, 1, 0))
    rows.append(dict(seed=SEEDS[j], zones_differ_own_calibration=int((z32 != z16).sum()), flag_status_differs=int(((z32 == 2) != (z16 == 2)).sum()),
                     zones_differ_fp16_cutoffs=int((zc != z16).sum()), max_abs_logit_diff=float(np.abs(F[c].to_numpy() - vt[c].to_numpy()).max()),
                     median_abs_logit_diff=float(np.median(np.abs(F[c].to_numpy() - vt[c].to_numpy())))))
s0 = pd.DataFrame(rows); s0['serve_from_fp32'] = s0.zones_differ_own_calibration > 10
s0.to_csv(R + '22_fp16_vs_fp32_zones.csv', index=False)
print(s0.to_string(index=False)); print('\nserve from fp32:', s0.loc[s0.serve_from_fp32, 'seed'].tolist())

 seed  zones_differ_own_calibration  flag_status_differs  zones_differ_fp16_cutoffs  max_abs_logit_diff  median_abs_logit_diff  serve_from_fp32
   42                             1                    1                          2            0.083984                    0.0            False
   43                             0                    0                          0            0.064941                    0.0            False
   44                             0                    0                          0            0.083984                    0.0            False
   45                             0                    0                          0            0.054688                    0.0            False
   46                             0                    0                          0            0.092774                    0.0            False
   47                             0                    0                          0            0.066406                    0.0          

## 3. The official verdict, rebuilt on the fp16 copies

The official ensemble is the mean of seeds 42–44's p-values, re-calibrated on the validation humans (`src/ensemble.PEnsemble`), flagged when its p ≤ 0.00235.
Here it is recomputed from the fp16 copies, so each essay can be compared with its jurors. As a check, on the held-out slices it is compared with the pre-registered scores of notebook 20 (computed on the training path).

In [4]:
ens = PEnsemble(np.column_stack([V[c] for c in COLS[:3]]))
p_ens = ens.p_ens(L[COLS[:3]].to_numpy())
ens_flag = p_ens <= TAU_FLAG
h20 = pd.read_csv(R + '20_heldout_scores.csv').set_index('row_id')
common = h20.index.intersection(L.index)
off = h20.loc[common, 'p_ens'] <= TAU_FLAG
mine = pd.Series(ens_flag, index=L.index).loc[common]
print('held-out essays:', len(common), '| flagged in 20:', int(off.sum()), '| flagged here:', int(mine.sum()), '| verdict differs on', int((off != mine).sum()))

held-out essays: 9856 | flagged in 20: 5294 | flagged here: 5294 | verdict differs on 0


## 4. S1, S3 — how many of the 20 jurors flag each essay?

Step: count, for each essay, the jurors whose zone is "flagged". We show the whole distribution (0 to 20) separately for humans and AI in each population, then the summary shares. Group B (seeds 45–61, 17 jurors) is reported separately as declared.

In [5]:
M2 = M.copy()
M2['n_all'] = FL.sum(1); M2['n_A'] = FL[:, :3].sum(1); M2['n_B'] = FL[:, 3:].sum(1)
M2['n_above_all'] = (Z == 1).sum(1)
M2['ens_p'] = p_ens; M2['ens_flagged'] = ens_flag
pops = {'test_humans': (M2.split == 'test') & (M2.label == 0), 'test_AI': (M2.split == 'test') & (M2.label == 1),
        'heldout_prompt_humans': (M2.split == 'heldout_prompt') & (M2.label == 0), 'heldout_prompt_AI': (M2.split == 'heldout_prompt') & (M2.label == 1),
        'heldout_generator_AI [descriptive]': (M2.split == 'heldout_generator')}
for p in ['Does the electoral college work?', 'Summer projects']:
    pops[f'heldout_prompt_humans: {p}'] = pops['heldout_prompt_humans'] & (M2.prompt_name == p)
tabs = {}
for pn, mk in pops.items():
    for g, idx in GROUPS.items():
        col = {'all20': 'n_all', 'A_42-44': 'n_A', 'B_45-61': 'n_B'}[g]; k = len(idx); n = int(mk.sum())
        cnt = M2.loc[mk, col].value_counts().reindex(range(k + 1), fill_value=0)
        tabs[(pn, g)] = cnt
        for v, c in cnt.items():
            r, lo, hi = rate(c, n); add('S1', pn, g, f'n_flag={v}', r, lo, hi, n)
        is_h = 'human' in pn
        ge1 = (M2.loc[mk, col] >= 1).sum(); allk = (M2.loc[mk, col] == k).sum(); maj = (M2.loc[mk, col] > k / 2).sum()
        for name, c in (('share_ge1', ge1), ('share_all', allk), ('share_majority', maj)):
            r, lo, hi = rate(c, n); add('S3', pn, g, name, r, lo, hi, n)
def show(pn, g='all20'):
    c = tabs[(pn, g)]; print(f'{pn} [{g}] n={c.sum()}:', {int(k): int(v) for k, v in c.items() if v})
for pn in pops: show(pn)
t = pd.DataFrame([{**{'population': pn, 'group': g}, **{m: summary_row['value'] for summary_row in summary if summary_row['stat']=='S3' and summary_row['population']==pn and summary_row['group']==g for m in [summary_row['metric']]}} for pn in pops for g in GROUPS])
print((t.set_index(['population', 'group']) * 100).round(2).to_string())

test_humans [all20] n=2291: {0: 2280, 1: 3, 2: 3, 3: 1, 5: 1, 10: 1, 19: 1, 20: 1}
test_AI [all20] n=1210: {0: 7, 1: 1, 11: 1, 12: 1, 15: 1, 16: 2, 18: 1, 19: 7, 20: 1189}
heldout_prompt_humans [all20] n=4464: {0: 4373, 1: 31, 2: 5, 3: 9, 4: 4, 5: 2, 6: 4, 7: 3, 8: 6, 9: 1, 10: 4, 12: 3, 13: 2, 14: 2, 15: 1, 16: 2, 17: 1, 18: 1, 19: 5, 20: 5}
heldout_prompt_AI [all20] n=2671: {0: 4, 2: 1, 6: 2, 9: 1, 10: 3, 12: 2, 14: 1, 15: 4, 16: 4, 17: 1, 18: 1, 19: 27, 20: 2620}
heldout_generator_AI [descriptive] [all20] n=2721: {0: 8, 1: 3, 2: 11, 3: 8, 4: 10, 5: 10, 6: 8, 7: 8, 8: 9, 9: 12, 10: 15, 11: 16, 12: 20, 13: 17, 14: 29, 15: 39, 16: 45, 17: 76, 18: 87, 19: 185, 20: 2105}
heldout_prompt_humans: Does the electoral college work? [all20] n=2714: {0: 2700, 1: 9, 3: 2, 6: 1, 14: 1, 19: 1}
heldout_prompt_humans: Summer projects [all20] n=1750: {0: 1673, 1: 22, 2: 5, 3: 7, 4: 4, 5: 2, 6: 3, 7: 3, 8: 6, 9: 1, 10: 4, 12: 3, 13: 2, 14: 1, 15: 1, 16: 2, 17: 1, 18: 1, 19: 4, 20: 5}
                  

### Humans that the official ensemble flags (S2)
Of the true student essays that the official ensemble wrongly flags, how many of the 20 jurors agree with the flag? Seeds 42–44 are inside the ensemble, so group B is the independent view.

In [6]:
mk = ens_flag & (M2.label == 0) & M2.split.isin(['test', 'heldout_prompt'])
sub = M2[mk]
print('humans flagged by the official ensemble (test + held-out prompts):', len(sub))
for col, k in (('n_all', 20), ('n_B', 17)):
    cnt = sub[col].value_counts().reindex(range(k + 1), fill_value=0)
    print(col, {int(a): int(b) for a, b in cnt.items() if b}, '| mean', round(sub[col].mean(), 2))
    for v, c in cnt.items(): add('S2', 'ensemble_flagged_humans(test+heldout_prompt)', col, f'n_flag={v}', c / len(sub), np.nan, np.nan, len(sub))
    add('S2', 'ensemble_flagged_humans(test+heldout_prompt)', col, 'mean_n_flag', sub[col].mean(), n=len(sub))
sub[['split', 'prompt_name', 'n_all', 'n_B']].sort_values('n_all').head(12)

humans flagged by the official ensemble (test + held-out prompts): 33
n_all {4: 1, 6: 2, 8: 3, 9: 1, 10: 3, 12: 2, 13: 2, 14: 2, 15: 1, 16: 2, 17: 1, 18: 1, 19: 6, 20: 6} | mean 14.33
n_B {3: 1, 4: 1, 5: 3, 6: 1, 7: 2, 8: 2, 9: 2, 10: 1, 11: 1, 12: 3, 13: 2, 14: 1, 15: 1, 16: 6, 17: 6} | mean 11.7


,split,prompt_name,n_all,n_B
row_id,,,,
3513,heldout_prompt,Summer projects,4,3
3588,heldout_prompt,Summer projects,6,4
43016,heldout_prompt,Does the electoral college work?,6,5
3545,heldout_prompt,Summer projects,8,6
3853,heldout_prompt,Summer projects,8,5
3818,heldout_prompt,Summer projects,8,5
3131,heldout_prompt,Summer projects,9,7
3166,heldout_prompt,Summer projects,10,8
4236,heldout_prompt,Summer projects,10,7


## 5. S4 — each juror's detection of AI at its own flag line

Flag lines sit at logit 7.3–9.7 while the head saturates at about 9.8, so a juror may miss AI essays that sit just under its line. We list, for each juror, the share of AI essays it flags (test, held-out prompts, held-out generators) and its false flags on humans.

In [7]:
rows = []
for j, s in enumerate(SEEDS):
    d = dict(seed=s, group='A' if s < 45 else 'B', flag_line=flag_line[j])
    for name, mk in (('test_AI', pops['test_AI']), ('heldout_prompt_AI', pops['heldout_prompt_AI']), ('heldout_generator_AI', pops['heldout_generator_AI [descriptive]']),
                     ('test_humans', pops['test_humans']), ('heldout_prompt_humans', pops['heldout_prompt_humans'])):
        k = int(FL[mk.to_numpy(), j].sum()); n = int(mk.sum()); r, lo, hi = rate(k, n)
        d[name] = r; d[name + '_lo'] = lo; d[name + '_hi'] = hi
    rows.append(d)
pj = pd.DataFrame(rows)
for name in ['test_AI', 'heldout_prompt_AI', 'heldout_generator_AI', 'test_humans', 'heldout_prompt_humans']:
    for g, sel in (('all20', pj.index), ('A_42-44', pj.group == 'A'), ('B_45-61', pj.group == 'B')):
        v = pj.loc[sel, name]
        for m, f in (('min', v.min()), ('median', v.median()), ('max', v.max())): add('S4', name, g, f'detect_or_flag_rate_{m}_over_jurors', f)
print((pj.set_index('seed')[['group', 'flag_line', 'test_AI', 'heldout_prompt_AI', 'heldout_generator_AI', 'test_humans', 'heldout_prompt_humans']] * [1, 1, 100, 100, 100, 100, 100]).round(2).to_string())

     group  flag_line  test_AI  heldout_prompt_AI  heldout_generator_AI  test_humans  heldout_prompt_humans
seed                                                                                                       
42       A       9.09    99.34              99.51                 96.66         0.13                   0.87
43       A       7.27    99.17              99.70                 95.37         0.17                   0.74
44       A       9.38    98.84              99.44                 93.50         0.13                   0.69
45       B       8.78    99.42              99.70                 97.87         0.22                   0.87
46       B       8.43    99.34              99.70                 97.54         0.13                   0.54
47       B       8.40    99.34              99.78                 99.30         0.22                   1.03
48       B       9.30    99.17              99.51                 97.83         0.13                   0.60
49       B       9.33    99.

## 6. S5 — when the essay is clearly AI, does the jury still split?

For AI essays we print the distribution of the number of jurors flagging. Then the *ceiling* essays: AI essays on which all 20 jurors score at least 8.0 (the head saturates near 9.8), i.e. essays no juror finds ambiguous. If these still split the jury, the page has to say so.

In [8]:
ai = M2.split.isin(['test', 'heldout_prompt', 'heldout_generator']) & (M2.label == 1)
Lm = L.to_numpy()
ceil_mask = ai.to_numpy() & (Lm.min(1) >= 8.0)
print('AI essays (test + both held-out slices):', int(ai.sum()), '| every juror >= 8.0:', int(ceil_mask.sum()))
for pn in ['test_AI', 'heldout_prompt_AI', 'heldout_generator_AI [descriptive]']:
    mk = pops[pn].to_numpy()
    cm = mk & (Lm.min(1) >= 8.0)
    k = int((M2.n_all[cm] < 20).sum()); n = int(cm.sum()); r, lo, hi = rate(k, max(n, 1))
    add('S5', pn, 'all20', 'ceiling_ai_not_unanimous', r, lo, hi, n); add('S5', pn, 'all20', 'ceiling_ai_count', n)
    print(pn, '| ceiling essays', n, '| not unanimous', k, f'({100*r:.1f}%)', '| their n_flag:', M2.n_all[cm].value_counts().sort_index().to_dict())
allai = ai.to_numpy()
print('all AI: n_flag==20:', int((M2.n_all[allai] == 20).sum()), '| 17..19:', int(M2.n_all[allai].between(17, 19).sum()), '| 11..16:', int(M2.n_all[allai].between(11, 16).sum()),
      '| 1..10:', int(M2.n_all[allai].between(1, 10).sum()), '| 0:', int((M2.n_all[allai] == 0).sum()))
# how many different jurors miss the clear AI essays?
miss = (~FL)[ceil_mask]
print('ceiling essays where some juror does not flag: which jurors miss them (count of essays):', dict(zip(SEEDS, miss.sum(0).tolist())))

AI essays (test + both held-out slices): 6602 | every juror >= 8.0: 6204
test_AI | ceiling essays 1199 | not unanimous 10 (0.8%) | their n_flag: {16: 2, 18: 1, 19: 7, 20: 1189}
heldout_prompt_AI | ceiling essays 2648 | not unanimous 29 (1.1%) | their n_flag: {15: 1, 16: 3, 18: 1, 19: 24, 20: 2619}
heldout_generator_AI [descriptive] | ceiling essays 2357 | not unanimous 254 (10.8%) | their n_flag: {14: 1, 15: 2, 16: 6, 17: 17, 18: 54, 19: 174, 20: 2103}
all AI: n_flag==20: 5914 | 17..19: 385 | 11..16: 182 | 1..10: 102 | 0: 19
ceiling essays where some juror does not flag: which jurors miss them (count of essays): {42: 1, 43: 0, 44: 13, 45: 0, 46: 1, 47: 0, 48: 2, 49: 8, 50: 2, 51: 3, 52: 203, 53: 0, 54: 7, 55: 2, 56: 17, 57: 2, 58: 134, 59: 0, 60: 33, 61: 5}


## 7. S6 — how far below its ceiling each juror's flag line sits, and who lives in that gap

*Ceiling* = the juror's highest logit over validation and test. *Gap* = ceiling − flag line. A *fragile band* = within 0.5 logit of the flag line on either side. A typo, a rounding step or one validation essay that moves the line can flip a vote there.

In [9]:
test_idx = (M2.split == 'test').to_numpy(); lab = M2.label.to_numpy()
rows = []
for j, s in enumerate(SEEDS):
    x = Lm[:, j]; xt = x[test_idx]; lt = lab[test_idx]
    ceiling = x[(M2.split.isin(['val', 'test'])).to_numpy()].max()
    band = np.abs(xt - flag_line[j]) <= 0.5
    above = (xt > flag_line[j]) & (xt <= flag_line[j] + 0.5)
    rows.append(dict(seed=s, ceiling=ceiling, flag_line=flag_line[j], gap=ceiling - flag_line[j],
                     test_humans_in_band=int((band & (lt == 0)).sum()), test_AI_in_band=int((band & (lt == 1)).sum()),
                     test_AI_flagged=int(((xt > flag_line[j]) & (lt == 1)).sum()), test_AI_flagged_by_less_than_0p5=int((above & (lt == 1)).sum()),
                     test_AI_below_line_within_0p5=int((band & (xt <= flag_line[j]) & (lt == 1)).sum())))
gp = pd.DataFrame(rows); gp['share_flagged_AI_within_0p5_of_line'] = gp.test_AI_flagged_by_less_than_0p5 / gp.test_AI_flagged
pj = pj.merge(gp.drop(columns='flag_line'), on='seed')
for m in ['gap', 'test_humans_in_band', 'test_AI_in_band', 'share_flagged_AI_within_0p5_of_line']:
    for g, sel in (('all20', gp.index), ('A_42-44', gp.seed < 45), ('B_45-61', gp.seed >= 45)):
        v = gp.loc[sel, m]
        for nm, f in (('min', v.min()), ('median', v.median()), ('max', v.max())): add('S6', 'test', g, f'{m}_{nm}_over_jurors', f)
print(gp.round(3).to_string(index=False))

 seed  ceiling  flag_line   gap  test_humans_in_band  test_AI_in_band  test_AI_flagged  test_AI_flagged_by_less_than_0p5  test_AI_below_line_within_0p5  share_flagged_AI_within_0p5_of_line
   42    9.766      9.086 0.680                    4               19             1202                                18                              1                                0.015
   43    9.609      7.266 2.344                    2                3             1200                                 0                              3                                0.000
   44    9.648      9.383 0.266                    7             1203             1196                              1196                              7                                1.000
   45    9.914      8.781 1.133                    1                3             1203                                 3                              0                                0.002
   46    9.648      8.430 1.219                    4   

## 8. S7 — do the jurors rank essays the same way?

Spearman rank correlation between two jurors' logits: 1 means the same order of essays. Here for every pair of jurors on validation and on test, and within humans and within AI (where the order is not just "AI above human").

In [10]:
def spear(mask):
    X = Lm[mask]; Rk = np.apply_along_axis(rankdata, 0, X); C = np.corrcoef(Rk, rowvar=False); return C
res = {}
for nm, mk in (('val', M2.split == 'val'), ('test', M2.split == 'test'), ('test_humans', pops['test_humans']), ('test_AI', pops['test_AI']),
               ('val_humans', M2.split.eq('val') & M2.label.eq(0)), ('val_AI', M2.split.eq('val') & M2.label.eq(1))):
    C = spear(mk.to_numpy()); res[nm] = C
    iu = np.triu_indices(20, 1)
    for g, sel in (('all20', np.ones(len(iu[0]), bool)), ('A_42-44', (iu[0] < 3) & (iu[1] < 3)), ('B_45-61', (iu[0] >= 3) & (iu[1] >= 3)), ('A_vs_B', (iu[0] < 3) & (iu[1] >= 3))):
        v = C[iu][sel]
        for m, f in (('mean', v.mean()), ('min', v.min()), ('max', v.max())): add('S7', nm, g, f'spearman_{m}', float(f), n=int(sel.sum()))
    print(f'{nm:12s} all pairs: mean {C[iu].mean():.4f} min {C[iu].min():.4f} | within B: mean {C[3:,3:][np.triu_indices(17,1)].mean():.4f}')

val          all pairs: mean 0.9542 min 0.9241 | within B: mean 0.9545


test         all pairs: mean 0.9529 min 0.9226 | within B: mean 0.9532
test_humans  all pairs: mean 0.8921 min 0.7828 | within B: mean 0.8955
test_AI      all pairs: mean 0.5824 min 0.2523 | within B: mean 0.5673
val_humans   all pairs: mean 0.8990 min 0.8025 | within B: mean 0.9023
val_AI       all pairs: mean 0.5642 min 0.2212 | within B: mean 0.5496


## 9. S8 — how much does one juror's flag line move when its validation humans are resampled?

The flag line is the 5th-highest score among about 2,300 validation humans, so it rests on very few essays. We resample the validation humans 1,000 times (whole `dup_cluster` groups at a time, as the earlier notebooks do), recompute the juror's cut-off and count how many test + held-out-prompt essays change between flagged and not flagged from that alone.
Jurors: seed 42 (ensemble member), 50 (middle), 56 (lowest val-1% cut-off).

In [11]:
rng = np.random.default_rng(42)
vh_ids = val_h[val_h].index; vclu = M2.loc[vh_ids, 'dup_cluster'].to_numpy()
cl_ids, inv = np.unique(vclu, return_inverse=True)
members = [np.flatnonzero(inv == i) for i in range(len(cl_ids))]
pool = M2.split.isin(['test', 'heldout_prompt']).to_numpy(); pool_lab = lab[pool]
rows = []
for s in (42, 50, 56):
    j = SEEDS.index(s); v = V[COLS[j]]; x = Lm[pool, j]
    base = tail_p(x, v) <= TAU_FLAG
    cut, chg_h, chg_a, to_flag, to_unflag = [], [], [], [], []
    for b in range(1000):
        pick = rng.integers(0, len(members), len(members)); ii = np.concatenate([members[i] for i in pick]); vb = v[ii]
        desc = np.sort(vb)[::-1]; cut.append(desc[int(np.floor(TAU_FLAG * (1 + len(vb)))) - 1])
        new = tail_p(x, vb) <= TAU_FLAG
        chg_h.append(int((new != base)[pool_lab == 0].sum())); chg_a.append(int((new != base)[pool_lab == 1].sum()))
        to_flag.append(int((new & ~base).sum())); to_unflag.append(int((~new & base).sum()))
    cut = np.array(cut); tot = np.array(chg_h) + np.array(chg_a)
    rows.append(dict(seed=s, flag_line=flag_line[j], cutoff_p2_5=np.percentile(cut, 2.5), cutoff_median=np.median(cut), cutoff_p97_5=np.percentile(cut, 97.5),
                     essays_changing_mean=tot.mean(), essays_changing_p95=np.percentile(tot, 95), humans_changing_mean=np.mean(chg_h), AI_changing_mean=np.mean(chg_a),
                     newly_flagged_mean=np.mean(to_flag), newly_unflagged_mean=np.mean(to_unflag), pool_size=int(pool.sum())))
fb = pd.DataFrame(rows); fb.to_csv(R + '22_flagline_bootstrap.csv', index=False)
for r in rows:
    for m in ('cutoff_p2_5', 'cutoff_p97_5', 'essays_changing_mean', 'essays_changing_p95'): add('S8', 'test+heldout_prompt', f"seed{r['seed']}", m, r[m], n=r['pool_size'])
print(fb.round(3).to_string(index=False))

 seed  flag_line  cutoff_p2_5  cutoff_median  cutoff_p97_5  essays_changing_mean  essays_changing_p95  humans_changing_mean  AI_changing_mean  newly_flagged_mean  newly_unflagged_mean  pool_size
   42      9.086        7.949          9.086         9.328                11.756                 53.0                 9.064             2.692               8.391                 3.365      10636
   50      9.234        8.023          9.234         9.438                16.979                 70.0                14.582             2.397              14.501                 2.478      10636
   56      8.312        6.355          8.312         8.641                17.315                 61.0                13.994             3.321              15.171                 2.144      10636


## 10. S9 — typo jitter

Fixed sample (random seed 0) from test + held-out-prompt essays: up to 600 that **split the jury** (between 1 and 19 jurors flag them: "near the line") and 600 that do not (0 or 20: "far").
Each gets 5% typos with `inject_typos` (seed = row_id) and is re-scored by all 20 fp16 jurors. We count how often each juror's zone changes. The typo-set file is written once; the scoring is done by `scripts/jury_score_extra.py typos`.

In [12]:
import os
tpath = R + '22_typo_set.csv'
pool_ids = M2.index[M2.split.isin(['test', 'heldout_prompt']).to_numpy()]
rs = np.random.default_rng(0)
near_all = pool_ids[((M2.loc[pool_ids, 'n_all'] >= 1) & (M2.loc[pool_ids, 'n_all'] <= 19)).to_numpy()]
far_all = pool_ids[M2.loc[pool_ids, 'n_all'].isin([0, 20]).to_numpy()]
near = rs.choice(near_all, min(600, len(near_all)), replace=False); far = rs.choice(far_all, 600, replace=False)
ts = pd.DataFrame({'row_id': np.concatenate([near, far]), 'group': ['near'] * len(near) + ['far'] * len(far)})
print('essays that split the jury in test + held-out prompts:', len(near_all), '-> sampled', len(near), '| unanimous:', len(far_all), '-> sampled', len(far))
if not os.path.exists(tpath): ts.to_csv(tpath, index=False)
assert (pd.read_csv(tpath).row_id.to_numpy() == ts.row_id.to_numpy()).all()
assert os.path.exists(R + '22_juror_scores_typos5.csv'), 'run: python scripts/jury_score_extra.py typos  (then re-run this notebook)'

essays that split the jury in test + held-out prompts: 157 -> sampled 157 | unanimous: 10479 -> sampled 600


In [13]:
T = pd.read_csv(R + '22_juror_scores_typos5.csv').set_index('row_id')[COLS]
ts = ts.set_index('row_id'); assert (T.index == ts.index).all()
Pt = np.column_stack([tail_p(T[c].to_numpy(), V[c]) for c in COLS]); Zt = np.where(Pt <= TAU_FLAG, 2, np.where(Pt <= TAU_ABOVE, 1, 0))
Z0 = Z[M2.index.get_indexer(T.index)]
changed = (Zt != Z0); flag_changed = ((Zt == 2) != (Z0 == 2))
ens_t = PEnsemble(np.column_stack([V[c] for c in COLS[:3]])).p_ens(T[COLS[:3]].to_numpy()) <= TAU_FLAG
ens_0 = M2.loc[T.index, 'ens_flagged'].to_numpy()
labs = M2.loc[T.index, 'label'].to_numpy()
rows = []
for g in ('near', 'far'):
    for lb, lbn in ((0, 'humans'), (1, 'AI')):
        mk = (ts.group.to_numpy() == g) & (labs == lb); n = int(mk.sum())
        if n == 0: continue
        per_juror = changed[mk].mean(0); per_juror_flag = flag_changed[mk].mean(0)
        d = dict(group=g, label=lbn, n=n, mean_jurors_zone_changes=changed[mk].sum(1).mean(), mean_jurors_flag_status_changes=flag_changed[mk].sum(1).mean(),
                 share_essays_with_any_flag_change=(flag_changed[mk].sum(1) > 0).mean(), juror_zone_change_rate_min=per_juror.min(), juror_zone_change_rate_median=np.median(per_juror),
                 juror_zone_change_rate_max=per_juror.max(), mean_n_flag_before=FL[M2.index.get_indexer(T.index)][mk].sum(1).mean(), mean_n_flag_after=(Zt == 2)[mk].sum(1).mean(),
                 ensemble_verdict_changes=int((ens_t != ens_0)[mk].sum()), ensemble_verdict_change_share=(ens_t != ens_0)[mk].mean())
        rows.append(d)
        for m, v in d.items():
            if m not in ('group', 'label', 'n'): add('S9', f'typo5_{g}', lbn, m, v, n=n)
tj = pd.DataFrame(rows); tj.to_csv(R + '22_typo_jitter.csv', index=False)
print(tj.round(3).T.to_string())
pd.DataFrame({'seed': SEEDS, 'zone_change_rate_near': changed[ts.group.to_numpy() == 'near'].mean(0), 'zone_change_rate_far': changed[ts.group.to_numpy() == 'far'].mean(0)}).round(3).T

                                        0       1       2       3
group                                near    near     far     far
label                              humans      AI  humans      AI
n                                      96      61     370     230
mean_jurors_zone_changes           15.625  13.656   0.268   4.583
mean_jurors_flag_status_changes     5.573  11.311     0.0   4.583
share_essays_with_any_flag_change     1.0   0.984     0.0     0.6
juror_zone_change_rate_min          0.583   0.279   0.003   0.013
juror_zone_change_rate_median       0.781   0.746   0.011   0.276
juror_zone_change_rate_max          0.938   0.869   0.024   0.509
mean_n_flag_before                  5.719  16.148     0.0  19.913
mean_n_flag_after                   0.167   4.836     0.0   15.33
ensemble_verdict_changes               27      39       0      31
ensemble_verdict_change_share       0.281   0.639     0.0   0.135


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
seed,42.000,43.000,44.000,45.000,46.000,47.000,48.000,49.000,50.000,51.000,52.000,53.000,54.000,55.000,56.000,57.000,58.000,59.000,60.000,61.000
zone_change_rate_near,0.688,0.688,0.790,0.739,0.465,0.764,0.796,0.783,0.701,0.828,0.580,0.783,0.860,0.694,0.739,0.822,0.752,0.745,0.790,0.854
zone_change_rate_far,0.037,0.053,0.138,0.033,0.015,0.120,0.103,0.083,0.020,0.120,0.152,0.158,0.123,0.068,0.082,0.118,0.128,0.018,0.142,0.208


## 10b. Extra (not on the declared list): does jury agreement say anything about whether the official flag is right?
Among essays the official ensemble flags (test + held-out prompts), how many are students' essays at each level of jury agreement? And a reconciliation: the held-out-generator detection of the official ensemble here (all essays) vs notebook 20's 94.92% (on-topic essays only?).

In [14]:
fl = M2[M2.ens_flagged & M2.split.isin(['test', 'heldout_prompt'])]
print('official flags:', len(fl), '| of them students:', int((fl.label == 0).sum()))
for lo, hi in ((19, 20), (16, 18), (11, 15), (0, 10)):
    s = fl[fl.n_all.between(lo, hi)]; k = int((s.label == 0).sum()); r, a, b = rate(k, len(s))
    add('EXTRA', 'ensemble_flagged(test+heldout_prompt)', 'all20', f'student_share_when_n_flag_{lo}_to_{hi}', r, a, b, len(s))
    print(f'  {lo:2d}-{hi:2d} jurors flag: {len(s):5d} essays, {k:3d} students ({100*r:.2f}% [{100*a:.2f}, {100*b:.2f}])')
g = M2[M2.split == 'heldout_generator']
print('official ensemble on held-out generators: all essays', round(100 * g.ens_flagged.mean(), 2), '| on-topic only', round(100 * g[g.on_topic if 'on_topic' in g else slice(None)].ens_flagged.mean(), 2) if False else '')
ot = data.loc[g.index, 'on_topic']
print('  on-topic essays only:', round(100 * g.ens_flagged[ot.to_numpy()].mean(), 2), '(n=%d)' % int(ot.sum()), '| notebook 20 reported 94.92')

official flags: 3894 | of them students: 33
  19-20 jurors flag:  3855 essays,  12 students (0.31% [0.18, 0.54])
  16-18 jurors flag:    13 essays,   4 students (30.77% [12.68, 57.63])
  11-15 jurors flag:    15 essays,   7 students (46.67% [24.81, 69.88])
   0-10 jurors flag:    11 essays,  10 students (90.91% [62.26, 98.38])
official ensemble on held-out generators: all essays 95.66 | on-topic only 
  on-topic essays only: 94.92 (n=2048) | notebook 20 reported 94.92


## 10c. Post-hoc diagnostic (written after 22b, not on the declared list): which AI essays lose their flag under typos?
Phase 2 found that 31 of 230 unanimous-AI essays lost the official flag under 5% typos (13.5%), while the random on-topic test AI sample of 22b (`22b_attack_final.ipynb`) lost only 12 of 469 (2.6%). The difference is not noise: it depends on who wrote the essay. Below, the same flip (flagged before, not flagged after 5% typos) by AI source, for the Phase 2 sample and for the 22b samples.

In [15]:
src = data['source']; ontop = data['on_topic']
ts_ = pd.read_csv(R + '22_typo_set.csv').set_index('row_id'); Tt = pd.read_csv(R + '22_juror_scores_typos5.csv').set_index('row_id')[COLS]
after = ens.p_ens(Tt[COLS[:3]].to_numpy()) <= TAU_FLAG
d2 = ts_.assign(after=after).join(M2[['split', 'label', 'ens_flagged']]).join(src).join(ontop)
a = d2[(d2.group == 'far') & (d2.label == 1)]
print('(a) Phase 2 unanimous-AI sample, flag lost under 5% typos, by split and on-topic:')
print(a.groupby(['split', 'on_topic']).agg(n=('after', 'size'), lost=('after', lambda x: int((~x).sum()))).to_string())
print('by source:'); print(a.groupby('source').agg(n=('after', 'size'), lost=('after', lambda x: int((~x).sum()))).sort_values('lost', ascending=False).head(6).to_string())
LG22b = pd.read_csv(R + '22b_attack_juror_logits.csv')
def ens_flag(df): return ens.p_ens(df[COLS[:3]].to_numpy()) <= TAU_FLAG
o_ = LG22b[LG22b.attack == 'original'].set_index('row_id'); t_ = LG22b[LG22b.attack == 'typos 5%'].set_index('row_id')
fb = pd.DataFrame({'before': ens_flag(o_), 'after': ens_flag(t_.loc[o_.index])}, index=o_.index).join(o_[['group']]).join(src)
fb = fb[fb.group.str.startswith('AI') & fb.before]
print('\n(b) 22b random samples, flagged essays that lose the flag under 5% typos, by source:')
print(fb.groupby(['group', 'source']).agg(flagged_before=('after', 'size'), lost=('after', lambda x: int((~x).sum()))).to_string())

(a) Phase 2 unanimous-AI sample, flag lost under 5% typos, by split and on-topic:
                           n  lost
split          on_topic           
heldout_prompt False      45     0
               True      115    29
test           False      43     3
               True       27     0
by source:
                       n  lost
source                        
kingki19_palm         49    25
mistral7binstruct_v2  24     3
darragh_claude_v7      9     1
radekgpt4              8     1
radek_500             16     1
chat_gpt_moth         27     1



(b) 22b random samples, flagged essays that lose the flag under 5% typos, by source:


                                                            flagged_before  lost
group                   source                                                  
AI, held-out generators darragh_claude_v6                              113    35
                        darragh_claude_v7                              110    40
                        falcon_180b_v1                                  61     7
AI, test                NousResearch/Llama-2-7b-chat-hf                 14     0
                        chat_gpt_moth                                   50     0
                        cohere-command                                  29     3
                        kingki19_palm                                   82     0
                        llama2_chat                                     62     1
                        llama_70b_v1                                    37     4
                        mistral7binstruct_v1                            51     0
                        mist

## 11. Save
`22_jury_per_essay.csv` has the row id, split, label and counts only (no text). `22_jury_per_juror.csv` is the per-juror table. `22_jury_summary.csv` holds every number above in long format.

In [16]:
pe = M2.reset_index()[['row_id', 'split', 'label', 'prompt_name', 'n_all', 'n_A', 'n_B', 'n_above_all', 'ens_p', 'ens_flagged']].rename(columns={'n_all': 'n_flag_20', 'n_A': 'n_flag_A', 'n_B': 'n_flag_B', 'n_above_all': 'n_above_usual_20', 'ens_p': 'ensemble_p', 'ens_flagged': 'ensemble_flagged'})
pe.to_csv(R + '22_jury_per_essay.csv', index=False)
pj.to_csv(R + '22_jury_per_juror.csv', index=False)
sm = pd.DataFrame(summary); sm.to_csv(R + '22_jury_summary.csv', index=False)
print(len(pe), 'essays;', len(pj), 'jurors;', len(sm), 'summary rows;', sm.stat.value_counts().to_dict())

16858 essays; 20 jurors; 620 summary rows; {'S1': 301, 'S7': 72, 'S3': 63, 'S4': 45, 'S2': 41, 'S9': 40, 'S6': 36, 'S8': 12, 'S5': 6, 'EXTRA': 4}


## Key takeaways

1. **fp16 serving is safe (S0).** No juror's zones differ on more than 10 of the 7,002 essays (largest: seed 56, 3 essays), so all 20 are served from fp16. The logits differ by up to 0.095 but the median is 0.
2. **The rebuilt official ensemble reproduces notebook 20 exactly**: 0 of 9,856 held-out verdicts differ.
3. **Students (S1, S3).** On test, 0.48% of student essays get a flag from at least one juror, 0.04% from all 20 (the official ensemble: 0.09%). On unseen prompts: 2.04% at least one, 0.11% all 20, official 0.69%; "Summer projects" alone 4.40% at least one.
4. **Why "flagged by at least one of 20" is not a fair rate on its own.** Each extra juror is another chance for an unlucky flag, so the rate only grows with the number of jurors (here three times the official ensemble's on unseen prompts). Use "how many of 20" and the official verdict, never the any-one rate.
5. **AI essays (S1, S3).** 98.3% of test AI and 98.1% of unseen-prompt AI are flagged by all 20; 99.3% / 99.6% by a majority. For **unseen generators (descriptive)**: 77.4% by all 20, 96.3% by a majority, 19 AI essays overall by none.
6. **Detection varies a lot between jurors only on unseen generators (S4).** Test AI: 98.6–99.4% per juror; unseen-prompt AI: 98.3–99.9%; unseen generators: 81.3% (seed 52) to 99.3%; the weakest are seeds 52, 58 and 60, which have the highest or tightest flag lines.
7. **Clear AI essays can still split the jury (S5).** Among AI essays on which all 20 jurors score at least 8.0, 0.8% (test), 1.1% (unseen prompts) and 10.8% (unseen generators) are not flagged by all 20. Seeds 52 (203 essays), 58 (134) and 60 (33) cause most of it. The page must say that a split does not mean "probably human".
8. **The flag line sits close to the ceiling for some jurors (S6).** Median gap 0.80 logit; seed 52 has 0.19 and seed 44 0.27. For those two, 99.5% of flagged test AI essays lie within 0.5 logit of the line, so "flagged" there means "reached the saturated score". Elsewhere a median of 14 test AI and 1–9 test human essays sit in the 0.5-logit fragile band.
9. **Jurors agree on who is a student, not on how AI-like an AI essay is (S7).** Mean Spearman 0.95 over all essays, 0.89 among humans, but only 0.58 among AI essays (lowest pair 0.25).
10. **Calibration noise is real (S8).** A juror's flag line from resampled validation humans moves by 7.9 to 9.3 logit (seed 42) and 6.4 to 8.6 (seed 56) in 95% of resamples; this alone changes the flagged status of 12–17 of 10,636 essays on average (95th percentile 53–70), mostly students becoming flagged.
11. **Typo jitter (S9; the sample is chosen by jury state, so these are not detection rates).** Only 157 of the 10,636 test and unseen-prompt essays split the jury (1.5%). With 5% typos on those, 14–16 of the 20 jurors change zone per essay; on essays that were unanimous, students stay put (no flag changes), but 60% of the unanimous-AI sample sees at least one juror's flag change, their mean flag count falls from 19.9 to 15.3 and the official verdict flips on 13.5% (31 of 230). Strict p-scale flag lines sit near the ceiling, so typos that lower an AI essay's score a little can lift it out of "flagged".
12. **What agreement tells you about an official flag (extra; small-sample observation, not used on the page).** See 10b: when 19–20 jurors flag, 12 of 3,855 flagged essays are students (0.31%); when 10 or fewer do, 10 of 11 are, but that is 11 essays in total and the page does not say it. Unanimity does not clear an essay either: 12 student essays drew 19–20 flags. Decision (2026-11-09): a split means "less sure", never "probably a student".

13. **The flag is lost mostly on unseen data (10c, post hoc).** Under 5% typos the official flag is lost by 0 of 27 on-topic *test* AI essays in the Phase 2 sample, but by 29 of 115 on-topic *held-out-prompt* essays (25 of 49 PaLM essays). In the random 22b samples, flagged *test* AI essays lose it in 0 to 4 per source (of 8 to 82), but unseen-generator essays lose it in 35 of 113 (Claude v6), 40 of 110 (Claude v7) and 7 of 61 (Falcon). The 13.5% in takeaway 11 therefore mixes familiar and unseen data and is sample-dependent; the page must quote 22b's random-sample figures, split by familiar and unseen.

**Two sources of disagreement, to be stated on the page:** the trained weights differ with the seed, and each juror's flag line rests on the few validation essays near it (S8).

**Proposed wording for the page when the jury splits on an essay it flags** (numbers to be read from `results/22_jury_summary.csv`, not typed): "14 of 20 detectors say AI; 6 do not. A split like this does not mean a student wrote it: among essays written by AI, about 1 in 57 splits the jury on familiar topics and about 1 in 4.5 when the AI model is one the detectors never saw. A split means 'less sure', never 'probably a student'."
**Honest summary sentence:** "Twenty equally accurate detectors almost always agree on student essays (0.04% are flagged by all 20 on familiar topics, 0.11% on unseen ones) and on AI essays (98% flagged by all 20), but the odd essay splits them, and a split is a sign the verdict is less certain, in either direction."

**For Phase 3:** `/api/meta` should expose the S1, S3, S5, S6 and S8 statistics from `results/22_jury_summary.csv`; the page must label held-out figures as descriptive and quote the "any one of 20" rate only next to the official rate.